# 01. 데이터 준비 및 검증

이 노트북에서는 농업 도메인 Tool Calling 학습을 위한 instruction 데이터를 준비하고 검증합니다.

## 목차
1. 환경 설정
2. 데이터 로드 및 통계
3. 데이터 검증
4. Train/Validation 분할
5. 데이터 시각화

## 1. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data_utils import (
    load_instruction_data,
    validate_conversation,
    validate_tool_call,
    prepare_datasets,
    get_dataset_statistics,
    AVAILABLE_TOOLS,
    SYSTEM_PROMPT
)

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("환경 설정 완료!")

## 2. 데이터 로드 및 통계

In [ ]:
# 데이터 로드
data_path = '../data/processed/instruction_data_sample.json'
data = load_instruction_data(data_path)

print(f"총 샘플 수: {len(data)}")
print(f"\n첫 번째 샘플 미리보기:")
print(json.dumps(data[0], indent=2, ensure_ascii=False)[:500] + "...")

In [ ]:
# Tool 분포 확인
tool_counts = Counter()
complexity_counts = Counter()

for item in data:
    tool = item.get('metadata', {}).get('tool_used', 'unknown')
    complexity = item.get('metadata', {}).get('complexity', 'unknown')
    
    # multi_tool인 경우 분리
    if ',' in tool:
        for t in tool.split(','):
            tool_counts[t.strip()] += 1
    else:
        tool_counts[tool] += 1
    
    complexity_counts[complexity] += 1

print("Tool 분포:")
for tool, count in tool_counts.most_common():
    print(f"  {tool}: {count}")

print(f"\n복잡도 분포:")
for complexity, count in complexity_counts.most_common():
    print(f"  {complexity}: {count}")

In [ ]:
# 시각화: Tool 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tool 분포 바 차트
tools = list(tool_counts.keys())
counts = list(tool_counts.values())

axes[0].barh(tools, counts, color='steelblue')
axes[0].set_xlabel('Count')
axes[0].set_title('Tool Distribution')
for i, v in enumerate(counts):
    axes[0].text(v + 0.5, i, str(v), va='center')

# 복잡도 분포 파이 차트
axes[1].pie(
    complexity_counts.values(),
    labels=complexity_counts.keys(),
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Complexity Distribution')

plt.tight_layout()
plt.savefig('../results/data_distribution.png', dpi=150)
plt.show()

## 3. 데이터 검증

In [ ]:
# 모든 대화 구조 검증
valid_samples = []
invalid_samples = []

for item in data:
    is_valid, error_msg = validate_conversation(item)
    
    if is_valid:
        valid_samples.append(item)
    else:
        invalid_samples.append({
            'id': item.get('id', 'unknown'),
            'error': error_msg
        })

print(f"유효한 샘플: {len(valid_samples)}/{len(data)}")
print(f"무효한 샘플: {len(invalid_samples)}/{len(data)}")

if invalid_samples:
    print("\n무효한 샘플 목록:")
    for sample in invalid_samples[:10]:  # 최대 10개만 표시
        print(f"  {sample['id']}: {sample['error']}")

In [ ]:
# Tool Call 형식 상세 검증
tool_call_validation = []

for item in data:
    for msg in item.get('conversations', []):
        if msg.get('role') == 'assistant' and '<tool_call>' in msg.get('content', ''):
            is_valid, error_msg, parsed = validate_tool_call(msg['content'])
            
            tool_call_validation.append({
                'id': item.get('id'),
                'valid': is_valid,
                'error': error_msg if not is_valid else None,
                'tool_name': parsed.get('name') if parsed else None
            })

# 결과 요약
df_validation = pd.DataFrame(tool_call_validation)
print(f"Tool Call 검증 결과:")
print(f"  유효: {df_validation['valid'].sum()}")
print(f"  무효: {(~df_validation['valid']).sum()}")

if (~df_validation['valid']).any():
    print("\n무효한 Tool Call:")
    print(df_validation[~df_validation['valid']])

## 4. Train/Validation 분할

In [ ]:
# 데이터셋 준비 (90% train, 10% validation)
train_data, val_data = prepare_datasets(
    data_path=data_path,
    output_dir='../data/processed',
    test_size=0.1,
    seed=42
)

print(f"\nTrain 데이터: {len(train_data)}개")
print(f"Validation 데이터: {len(val_data)}개")

In [ ]:
# 분할 후 Tool 분포 확인
def get_tool_distribution(data):
    tools = []
    for item in data:
        tool = item.get('metadata', {}).get('tool_used', 'unknown')
        if ',' in tool:
            tools.extend([t.strip() for t in tool.split(',')])
        else:
            tools.append(tool)
    return Counter(tools)

train_tools = get_tool_distribution(train_data)
val_tools = get_tool_distribution(val_data)

print("Train Tool 분포:")
for tool, count in train_tools.most_common():
    print(f"  {tool}: {count}")

print("\nValidation Tool 분포:")
for tool, count in val_tools.most_common():
    print(f"  {tool}: {count}")

## 5. 데이터 시각화

In [ ]:
# 대화 길이 분포
conversation_lengths = [len(item.get('conversations', [])) for item in data]

plt.figure(figsize=(10, 5))
plt.hist(conversation_lengths, bins=range(min(conversation_lengths), max(conversation_lengths) + 2), 
         edgecolor='black', alpha=0.7)
plt.xlabel('Number of Messages')
plt.ylabel('Count')
plt.title('Conversation Length Distribution')
plt.xticks(range(min(conversation_lengths), max(conversation_lengths) + 1))
plt.savefig('../results/conversation_length_dist.png', dpi=150)
plt.show()

print(f"평균 대화 길이: {sum(conversation_lengths)/len(conversation_lengths):.2f}")
print(f"최소: {min(conversation_lengths)}, 최대: {max(conversation_lengths)}")

In [ ]:
# 텍스트 길이 (토큰 수 추정)
from src.data_utils import format_conversation

text_lengths = []
for item in data:
    text = format_conversation(item)
    # 대략 4자당 1토큰으로 추정
    estimated_tokens = len(text) / 4
    text_lengths.append(estimated_tokens)

plt.figure(figsize=(10, 5))
plt.hist(text_lengths, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Estimated Tokens')
plt.ylabel('Count')
plt.title('Text Length Distribution (Estimated Tokens)')
plt.axvline(x=sum(text_lengths)/len(text_lengths), color='red', linestyle='--', label='Mean')
plt.legend()
plt.savefig('../results/text_length_dist.png', dpi=150)
plt.show()

print(f"평균 추정 토큰 수: {sum(text_lengths)/len(text_lengths):.0f}")
print(f"최소: {min(text_lengths):.0f}, 최대: {max(text_lengths):.0f}")

In [ ]:
# 최종 통계 저장
stats = get_dataset_statistics(data)

print("\n=== 데이터셋 최종 통계 ===")
for key, value in stats.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

# JSON으로 저장
with open('../results/data_statistics.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print("\n통계가 results/data_statistics.json에 저장되었습니다.")

## 다음 단계

데이터 준비가 완료되었습니다. 다음 노트북에서 QLoRA Fine-tuning을 진행합니다:
- `02_qlora_training.ipynb`